# 0.32 — Crypto '18 basket: the opposite regime to genAI

Second case study for the basket component (method in 0.30, coverage audit in 0.31).
Theme from 0.29: **Crypto '18**, promoted **2017-11-28** (the `bitcoin futures` fire);
the BLOK ETF followed on 2018-01-17 — 50 days later.

**Hypothesis** (from the genAI test): ChatGPT was 7 weeks old at promotion, so the
filing channel was *structurally silent* — nobody had filed yet. Bitcoin, in contrast,
had been in filings since ~2014. If the theme is slow-burn, the filing channel should
already be **usable at t₀**.

**Known handicaps for this theme** (a company universe, not a cryptoasset universe):
Coinbase still private in 2017, miners listed abroad (HIVE → TSX-V), Overstock —
2017's loudest pivot — unreachable through the ticker bridge (OSTK → BYON churn;
BYON doesn't resolve in SEC's map either).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import theme_basket as tb

pd.set_option("display.width", 200)

THEME = dict(
    vocab=['bitcoin', 'blockchain', 'cboe', 'cme', 'coinbase', 'crypto',
           'cryptocurrency', 'hive', 'ico', 'natixis', 'riot'],
    promote="2017-11-28", etf=("BLOK", "2018-01-17"))
VOCAB, PROMOTE = THEME["vocab"], THEME["promote"]

## 1 · Term specificity in the 2016–18 filing corpus

Same pre-flight as 0.30: how many 10-K/10-Qs mention each term in
`[promote − 365d, promote + 120d]`? Note the different failure surface vs genAI:
`riot` is insurance boilerplate ("war, riot, civil unrest"), `cme`/`cboe` are
derivatives boilerplate, `hive` collides with Apache Hive and bees.

In [2]:
startdt, enddt = tb.make_window(PROMOTE, 365, 120)
totals = pd.Series({t: tb.fts_total(t, startdt, enddt)[0] for t in VOCAB}, name="filings")
totals.sort_values().to_frame().assign(generic=lambda d: d.filings > 1000)

,filings,generic
coinbase,4,False
hive,46,False
ico,51,False
bitcoin,82,False
crypto,83,False
cryptocurrency,83,False
blockchain,201,False
cboe,224,False
riot,428,False
cme,665,False


## 2 · Point-in-time basket (promotion day, 2017-11-28)

Built by the CLI (`--name crypto_pit`); rebuilt here from cache if missing —
`build_basket` is idempotent and offline once cached.

In [3]:
PROC = Path.cwd().parent / "data" / "processed"

def load_or_build(name, lookahead):
    pq = PROC / f"theme_basket_{name}.parquet"
    if pq.exists():
        return pd.read_parquet(pq)
    return tb.build_basket(VOCAB, PROMOTE, lookback_days=365,
                           lookahead_days=lookahead)["basket"]

basket_pit = load_or_build("crypto_pit", 0)
show = ["ticker", "company", "n_filings", "total", "n_terms_hit", "per_1k_words",
        "bitcoin", "blockchain", "cryptocurrency", "cme", "cboe", "riot"]
basket_pit[show].head(15)

,ticker,company,n_filings,total,n_terms_hit,per_1k_words,bitcoin,blockchain,cryptocurrency,cme,cboe,riot
0,RIOT,"Riot Platforms, Inc.",1,283,6,4.163,22,122,49,0,0,85
1,BTCS,BTCS Inc.,5,698,4,6.825,581,110,0,0,0,0
2,GROW,U S GLOBAL INVESTORS INC,1,15,3,1.100,0,2,1,0,0,0
3,NDAQ,"NASDAQ, INC.",2,14,3,0.158,0,5,0,3,6,0
4,CME,CME GROUP INC.,4,658,2,5.835,0,0,0,650,8,0
5,CBOE,"Cboe Global Markets, Inc.",4,1070,2,3.363,0,0,0,6,1064,0
6,PIPR,PIPER SANDLER COMPANIES,1,4,2,0.674,0,0,0,2,2,0
7,BOIL,ProShares Trust II,4,185,2,0.540,0,0,0,29,156,0
8,SIVR,abrdn Silver ETF Trust,4,32,2,0.495,0,0,0,29,0,3
9,SLV,iShares Silver Trust,4,15,2,0.395,0,0,0,14,0,1


**Read the top rows**: Riot (pivoted to Riot Blockchain in Oct 2017, 283 mentions in
one filing) and BTCS top the *tradeable* basket on promotion day, with US Global
Investors (the HIVE backer), Nasdaq, CME and Cboe behind them. The filing channel is
already informative — nothing like genAI's empty point-in-time table.

## 3 · +120 days: FY2017 10-K season

In [4]:
basket_120 = load_or_build("crypto", 120)
basket_120[show].head(15)

,ticker,company,n_filings,total,n_terms_hit,per_1k_words,bitcoin,blockchain,cryptocurrency,cme,cboe,riot
0,BTCS,BTCS Inc.,6,1411,6,8.832,1170,214,8,1,0,0
1,RIOT,"Riot Platforms, Inc.",1,283,6,4.163,22,122,49,0,0,85
2,XYZ,"Block, Inc.",5,21,5,0.065,11,1,2,0,0,2
3,GROW,U S GLOBAL INVESTORS INC,2,57,4,1.841,1,5,22,0,0,0
4,NDAQ,"NASDAQ, INC.",3,35,4,0.206,1,15,0,5,14,0
5,CME,CME GROUP INC.,5,1028,3,5.964,1,0,0,1017,10,0
6,CBOE,"Cboe Global Markets, Inc.",5,1618,3,3.855,7,0,0,7,1604,0
7,ICE,"Intercontinental Exchange, Inc.",2,7,3,0.037,0,0,0,2,3,0
8,BX,Blackstone Inc.,5,10,3,0.020,0,1,0,0,6,3
9,BOIL,ProShares Trust II,5,257,2,0.547,0,0,0,38,219,0


In [5]:
# the regime-contrast statistic, comparable with 0.30's chatgpt/openai count
def core_mentioners(b):
    return int(((b.bitcoin > 0) | (b.blockchain > 0) | (b.cryptocurrency > 0)).sum())

print(f"companies mentioning bitcoin/blockchain/cryptocurrency:")
print(f"  point-in-time (2017-11-28): {core_mentioners(basket_pit)}   (genAI equivalent: 1)")
print(f"  +120 days                 : {core_mentioners(basket_120)}   (genAI equivalent: 13)")

# who enters the top-10 once FY2017 10-Ks land?
new = (set(basket_120.head(10).ticker) - set(basket_pit.head(10).ticker))
basket_120[basket_120.ticker.isin(new)][show]

companies mentioning bitcoin/blockchain/cryptocurrency:
  point-in-time (2017-11-28): 20   (genAI equivalent: 1)
  +120 days                 : 44   (genAI equivalent: 13)


,ticker,company,n_filings,total,n_terms_hit,per_1k_words,bitcoin,blockchain,cryptocurrency,cme,cboe,riot
2,XYZ,"Block, Inc.",5,21,5,0.065,11,1,2,0,0,2
7,ICE,"Intercontinental Exchange, Inc.",2,7,3,0.037,0,0,0,2,3,0
8,BX,Blackstone Inc.,5,10,3,0.020,0,1,0,0,6,3


**Block/Square is the genAI-style entrant**: `bitcoin` 0 → 11 once its FY2017 10-K
describes Cash App's bitcoin trading. But unlike genAI, the entrants *refine* an
already-good basket rather than create it.

## 4 · Failure modes specific to this theme

In [6]:
# 1) commodity trusts leak in via exchange words (cme/cboe), not crypto exposure
trusts = basket_120[(basket_120[["cme", "cboe"]].sum(axis=1) > 0)
                    & (basket_120[["bitcoin", "blockchain", "cryptocurrency"]].sum(axis=1) == 0)]
print(f"companies in basket via cme/cboe ONLY (no crypto word): {len(trusts)}")
print(trusts[["ticker", "company", "cme", "cboe", "total"]].head(8).to_string(index=False))

companies in basket via cme/cboe ONLY (no crypto word): 80
ticker                         company  cme  cboe  total
   ICE Intercontinental Exchange, Inc.    2     3      7
  BOIL              ProShares Trust II   38   219    257
  SIVR          abrdn Silver ETF Trust   36     0     39
   SLV            iShares Silver Trust   20     0     22
   IVR   Invesco Mortgage Capital Inc.   68     0     77
  HSTM                HEALTHSTREAM INC   30     0     31
  VIRT           Virtu Financial, Inc.   22    26     48
  PIPR         PIPER SANDLER COMPANIES    6     7     13


In [7]:
# 2) 'riot' is mostly insurance boilerplate: filings matching it vs Riot-the-company
hits = tb.fts_hits("riot", startdt, enddt)
print(f"'riot' filings in window: {len(hits)} — overwhelmingly \"war, riot, civil unrest\" clauses.")
print("Breadth-first ranking still surfaces Riot-the-company because it hits 6/11 terms.")

# 3) unreachable protagonists
cik_map = tb.load_cik_map()
for t in ("OSTK", "BYON", "COIN", "HIVE"):
    print(f"  {t}: {'resolves' if t in cik_map else 'NOT in SEC ticker map'}")

'riot' filings in window: 428 — overwhelmingly "war, riot, civil unrest" clauses.
Breadth-first ranking still surfaces Riot-the-company because it hits 6/11 terms.
  OSTK: NOT in SEC ticker map
  BYON: NOT in SEC ticker map
  COIN: resolves
  HIVE: resolves


## 5 · Takeaways

- **Regime confirmed**: slow-burn theme ⇒ the filing channel is *usable at t₀* —
  20 core-word mentioners point-in-time (vs genAI's 1), with the right names on top,
  50 days before the first ETF. Born-overnight themes need the news channel at t₀;
  slow-burn themes get a real filing basket immediately.
- **Vocabulary quality matters more here**: exchange words (`cme`, `cboe`) admit
  commodity trusts with zero crypto exposure; `riot`/`hive` are homonym-noisy. A v2
  scorer should weight *co-occurrence with core words* rather than raw counts.
- **Universe gaps are the binding constraint again**, in two distinct ways:
  Overstock is a *resolution* failure (OSTK→BYON churn, neither maps to a CIK);
  Coinbase and HIVE resolve in *today's* map but had **no SEC filings in 2017**
  (private / foreign-listed) — correctly absent point-in-time, and a reminder that
  resolving today ≠ tradeable then. Fixes: historical ticker→CIK mapping (0.31),
  possibly 20-F/foreign scope.

Reproduce:
```bash
python scripts/build_theme_basket.py --name crypto --as-of 2017-11-28 \
    --lookback 365 --lookahead 120 --vocab bitcoin blockchain cboe cme coinbase \
    crypto cryptocurrency hive ico natixis riot
# and --name crypto_pit --lookahead 0 for the point-in-time variant
```